In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import yfinance as yf

In [ ]:
msft = yf.Ticker("NVDA")
for key, value in msft.info.items():
    print(f"{key}: {value}")

In [ ]:
def get_prices(share_symbol, start_date, end_date, cache_filename):
    try:
        stock_prices = np.load(cache_filename)
    except IOError:
        ticker = yf.Ticker(share_symbol)
        stock_hist = ticker.history(start=start_date, end=end_date)
        stock_prices = stock_hist['Open'].values
        np.save(cache_filename, stock_prices)
    return stock_prices

In [ ]:
def plot_prices(prices):
    plt.title("open stock prices")
    plt.xlabel("day")
    plt.ylabel("price ($)")
    plt.plot(prices)
    plt.savefig('prices.png')

In [ ]:
prices = get_prices("NVDA", '2010-01-01', '2026-01-01', 'historical_stock_prices.npy')
plot_prices(prices)
print(prices)

In [ ]:
class DecisionPolicy:
    def select_action(self, current_state, step):
        pass
    def update_q(self, state, action, reward, next_state):
        pass

In [ ]:
import random

class RandomDesicionPolicy:
    def __init__ (self, actions):
        self.actions = actions
    def select_action( self, current_state, step):
        action = self.actions[random.randint(0, len(self.actions)-1)]
        return action
    def update_q(self, state, action, reward, next_state):
        pass

In [ ]:
def run_simulation(policy, initial_budget, initial_num_stocks, prices, hist, debug=False):
    budget = initial_budget
    num_stocks = initial_num_stocks
    share_value = 0
    transitions = list()
    
    for i in range(len(prices) - hist - 1):
        current_state = np.asmatrix(np.hstack((prices[i:i+hist], budget, num_stocks)))
        current_portfolio = budget + num_stocks * share_value
        action = policy.select_action(current_state, i)
        share_value = float(prices[i + hist + 1])
        
        if action == 'Buy' and budget >= share_value:
            budget -= share_value
            num_stocks += 1
        elif action == 'Sell' and num_stocks > 0:
            budget += share_value
            num_stocks -= 1
        else:
            action = 'Hold'
        
        new_portfolio = budget + num_stocks * share_value
        reward = new_portfolio - current_portfolio
        next_state = np.asmatrix(np.hstack((prices[i+1:i+hist+1], budget, num_stocks)))
        transitions.append((current_state, action, reward, next_state))
        policy.update_q(current_state, action, reward, next_state)
        portfolio = budget + num_stocks * share_value
        
        if debug:
            print('${}\t{} shares'.format(budget, num_stocks))
    
    return portfolio

In [ ]:
def run_simulations(policy, budget, num_stocks, prices, hist):
    num_tries = 60
    final_porfolios = list()
    
    for i in range(num_tries):
        if i % 100 == 0:
            print('Simulación {}/{} ({:.0f}%)'.format(i, num_tries, 100*i/num_tries))
        final_porfolio = run_simulation(policy, budget, num_stocks, prices, hist)
        final_porfolios.append(final_porfolio)
    
    avg, std = np.mean(final_porfolios), np.std(final_porfolios)
    return avg, std

In [ ]:
class QLearningDecisionPolicy(DecisionPolicy):
    def __init__(self, actions, input_dim):
        self.epsilon = 0.9
        self.gamma = 0.001
        self.actions = actions
        output_dim = len(actions)
        h1_dim = 200

        self.model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(input_dim,)),
            tf.keras.layers.Dense(h1_dim, activation='relu'),
            tf.keras.layers.Dense(output_dim, activation='relu')
        ])
        self.model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
            loss='mse'
        )

    def select_action(self, current_state, step):
        threshold = min(self.epsilon, step / 1000.)
        if random.random() < threshold:
            state = np.asarray(current_state).reshape(1, -1).astype(np.float32)
            action_q_vals = self.model(state, training=False).numpy()
            action_idx = np.argmax(action_q_vals)
            action = self.actions[action_idx]
        else:
            action = self.actions[random.randint(0, len(self.actions) - 1)]
        return action

    def update_q(self, state, action, reward, next_state):
        state = np.asarray(state).reshape(1, -1).astype(np.float32)
        next_state = np.asarray(next_state).reshape(1, -1).astype(np.float32)

        action_q_vals = self.model(state, training=False).numpy()
        next_action_q_vals = self.model(next_state, training=False).numpy()

        action_idx = self.actions.index(action)
        next_action_idx = np.argmax(next_action_q_vals)

        action_q_vals[0, action_idx] = (
            reward + self.gamma * next_action_q_vals[0, next_action_idx]
        )

        target = action_q_vals.astype(np.float32)
        self.model.train_on_batch(state, target)

In [ ]:
actions = ['Buy', 'Sell', 'Hold']
hist = 200
input_dim = hist + 2 
policy = QLearningDecisionPolicy(actions, input_dim)

avg, std = run_simulations(policy, budget=1000.0, num_stocks=0, prices=prices, hist=hist)
print(avg, std)